# Femboy ve futa LoRA'ları — MANUEL (ComfyUI arayüzünde elle) — Colab

ComfyUI'yi tünelle açar, grafiği **sen elle sürersin**. Uygulamanın koduna dokunmaz.

**Madde 501 · v11-4** *(Queen Editor v11 yol haritası)* — altı aday, tek oturumda: femboysXL ve Civitai'den iki femboy LoRA'sı daha, ve üç futa LoRA'sı; hepsi Queen Editor'ün Nova 3DCG ve DaSiWa'da çizdiği grafikle. Her gruptan seçilen LoRA sonra Queen Editor'de Nova ve DaSiWa'nın LoRA kutusuna girer.

**Gördüğün, Queen Editor'ün çizeceği.** Her adayın workflow'u Queen Editor'ün kendi fotoğraf grafiğinden (`queen-editor/assets/workflow_api.json`) kurulur: aynı node'lar, aynı adım, cfg ve FaceDetailer. LoRA yükleyicide **yalnız aday** var — panelde bir LoRA seçilince USNR'nin kapandığı gibi — ve adayın tetiği prompt'un **önünde**, Queen Editor'ün Slime'a yaptığı gibi. Tek fark negatif (5. bölüm).

**Input:** prompt (ComfyUI UI'da) · **Output:** resim

**Gerekenler:** GPU — **T4 yeter** (SDXL sınıfı model; A100 de olur, daha hızlı). Colab **Secrets**'ta **`CIVITAI_COOKIE`** (zorunlu — DaSiWa ve beş aday Civitai'den girişle iniyor) ve isteğe bağlı `GITHUB_TOKEN` (repo kapanırsa gerekir).

Sıra:
1. **CONFIG** — Civitai çerezi, GitHub token'ı + GPU
2. **Ortak yardımcılar** — log + fail-loud run + model doğrulama
3. **ComfyUI + 8 custom node** — grafiğin istediği paketler
4. **Modeller** — ~15,9 GB; önce boş disk, sonra Civitai erişimi; sonunda adayların eğitim etiketleri
5. **Workflow'lar** — her aday için bir tane, Queen Editor'ün grafiğinden
6. **Başlat + cloudflared tünel** → UI linki

> Drive kullanılmaz; modeller her oturumda kaynaktan iner (Colab geçici diski). Üretilen resimler de orada durur: beğendiğini UI'dan indir, runtime kapanınca gider.
>
> **Run all** → en alttaki linke gir → sol kenarda **Workflows** → bir aday → **Run** (Ctrl+Enter).
>
> Hangi dosya neden iniyor: **`indirilecekler.md`** · neyin nasıl karşılaştırılacağı ve prompt'lar: **`instructions.md`**

## 1) CONFIG

Doldurulacak bir şey yok → **Run all**.

**`CIVITAI_COOKIE`** Colab **Secrets**'tan okunur — queen-editor'ün okuduğu ad, tek sır bütün defterlere yeter. DaSiWa'nın ve beş adayın yazarı indirmeyi girişe bağlamış (girişsiz istek `401` dönüyor, 27 Ekim'de denendi); çerez yoksa ya da kırpılmışsa hücre burada, ilk saniyede durur, 15 GB indirdikten sonra değil. Değeri: civitai.red → giriş → F12 → Application → Cookies → `__Secure-civ-token` → değere **çift tıkla** → Ctrl+A → Ctrl+C (tek tıkla kopyalanan değer kırpık gelir). Ömrü ~30 gün. Değer hiçbir yere basılmaz.

**`GITHUB_TOKEN`** isteğe bağlı — queen-editor'ün defterinin okuduğu sır. 5. bölüm bu reponun iki dosyasını GitHub'dan okur: repo bugün açık, token'sız da iner. Token varsa GitHub'ın API'sinden iner, repo kapansa da. Değeri basılmaz.

GPU burada basılır; GPU yoksa hücre durur.

**Drive mount yok.** Bu deneme Drive kullanmıyor; standardın "önce Drive mount" kuralı Drive kullanan defterler için. **`HF_TOKEN` da yok:** Hugging Face'ten tek bir 52 MB'lık dosya iniyor (yüz dedektörü), token'sız.

In [ ]:
# === CONFIG ===
COMFY_PORT = 8188
COMFY_ROOT = "/content/ComfyUI"
COMFY_LOG  = "/content/comfyui.log"

import subprocess
from google.colab import userdata

# Required: DaSiWa's and five candidates' authors gate their downloads behind a login. Read here, so a
# missing or clipped cookie stops the run in its first second rather than after the first checkpoint.
# The name is queen-editor's: secrets belong to the Google account, so one paste serves every notebook
# (NOTEBOOK-STANDARD, section 4). The value is never printed.
COOKIE_VALUE = (userdata.get("CIVITAI_COOKIE") or "").strip()
# Two asserts, not one: a missing secret and a clipped value are different mistakes.
assert COOKIE_VALUE, "❌ CIVITAI_COOKIE okunamadı — Colab 🔑 Secrets'a ekle ve 'Notebook access'i aç"
assert len(COOKIE_VALUE) > 200, (
    f"❌ CIVITAI_COOKIE çok kısa ({len(COOKIE_VALUE)} karakter) — değer kırpılmış: civitai.red → F12 → "
    "Application → Cookies → __Secure-civ-token, değere çift tıkla → Ctrl+A → Ctrl+C")
print("✓ CIVITAI_COOKIE: Secrets'tan okundu")

# Optional: section 5 reads two files of this repo from GitHub. The repo is public today and they come
# without a token; with GITHUB_TOKEN -- the name queen-editor's notebook reads -- they come through
# GitHub's API, which reads a private repo too. The value is never printed.
try:
    GITHUB_TOKEN = (userdata.get("GITHUB_TOKEN") or "").strip()
except Exception as e:
    GITHUB_TOKEN = ""
    print(f"⚠️  GITHUB_TOKEN okunamadı — {type(e).__name__}: {e}\n"
          "    Grafikler token'sız inecek: repo açıkken yeter.")
if GITHUB_TOKEN:
    print("✓ GITHUB_TOKEN: Secrets'tan okundu")

# === GPU ===
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True)
except FileNotFoundError:
    smi = None
if smi is None or smi.returncode != 0 or not smi.stdout.strip():
    said = (smi.stderr or smi.stdout).strip() if smi else "nvidia-smi bulunamadı"
    raise RuntimeError(f"❌ GPU yok — Runtime → Change runtime type → T4 (ya da A100)\n{said}")
print(f"\n=== GPU ===\n{smi.stdout.strip()}")

## 2) Ortak Yardımcılar

Tek yerde tanımlanır, sonraki hücreler bunları kullanır:

- `log(msg, level)` — konsola durum satırı
- `run(cmd, label)` — **tek komut-hatası kapısı**: exit ≠ 0 veya timeout → `RuntimeError` + komutun **gerçek stderr** kuyruğu
- `check_safetensors(path)` — **ok** / **partial** / **invalid**. Beklenen boyutu **dosyanın kendi header'ından** hesaplar (tensör `data_offsets`); sunucuya sormaz
- `check_binary(path, floor)` — `.pt` / `.pth` için: kendi boyunu söylemeyen bir dosyada dürüst iki bakış — başı bir hata sayfası mı, ve bir taban boyu geçiyor mu
- `head_text(path)` — dosyanın ilk byte'ları, **ham**: bozuk bir indirmede orada ne olduğunu gösterir

İlk dördü Krea denemesinden (`krea-2/turbo.ipynb`), `check_binary` nova-3dcg denemesinden (`photo_generator/nova-3dcg/manual.ipynb`) birebir geliyor.

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
import os, json, time, struct, subprocess

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌", "SKIP": "⏭️ "}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr."""
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request (NOTEBOOK-STANDARD, section 3).
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

def check_binary(path, min_bytes):
    """State of a .pt/.pth model -> ("ok" | "partial" | "invalid", msg).

    Torch pickle/zip files carry no self-describing total length (unlike safetensors), so the
    honest cheap checks are: the head is not an HTML/JSON error page, and the size clears a loose
    floor (guards against truncated/error downloads, not an exact size).
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        head = f.read(16)
    if head[:1] in (b"<", b"{"):
        return "invalid", f"error page? ({human(size)})"
    if size < min_bytes:
        return "partial", f"{human(size)} < taban {human(min_bytes)}"
    return "ok", human(size)

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors, check_binary)")

## 3) ComfyUI + Custom Node'lar (8)

Workflow'lar nova-3dcg denemesinin arayüz grafiğinden kuruluyor (5. bölüm), ve o grafiğin node'ları bu sekiz paketten; kalanı ComfyUI'nin kendinde. Kapalı dalların paketleri de kurulur — UI grafiği açarken eksik node'u kırmızı gösterir. Listede Queen Editor'ün kurduğu on paketten ikisi yok: **ComfyUI-Manager** (eksik node beklenmiyor; 6. bölüm her node'u ComfyUI'ye sorar) ve **ComfyUI-DaSiWa-Nodes** (fotoğraf grafiği kullanmıyor).

Queen Editor'ün iki dersi burada da geçerli *(`queen-editor/colab/nodes.py`)*:
- **rgthree `main` dalından** klonlanır: 15 Ekim'de varsayılan dalı `dev` oldu, orada Power Lora Loader yok *(madde 470)*.
- **Impact-Pack'in listesinden `sam2` çıkarılır**: grafik SAM'ın ilk sürümünü yüklüyor, ve sam2'yi pip makinede derliyor — kullanıcının koşusunda 3 dakika *(madde 314)*.

Bir komut başarısız olursa hücre `RuntimeError` ile durur ve komutun kendi çıktısını basar.

In [ ]:
# === ComfyUI + the photo graph's custom nodes ===
import os

if not os.path.isdir(COMFY_ROOT):
    log("ComfyUI klonlanıyor")
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_ROOT],
        "clone ComfyUI", timeout=600)
run(["git", "pull", "-q"], "git pull ComfyUI", cwd=COMFY_ROOT)
log("ComfyUI gereksinimleri kuruluyor (birkaç dakika)")
run(["pip", "install", "-q", "-r", "requirements.txt", "opencv-python"], "pip install ComfyUI",
    cwd=COMFY_ROOT)

# (folder, repo, branch or None for its default): every pack a node of the UI graph comes from.
CUSTOM_NODES = [
    # Power Lora Loader, Seed, Image Comparer, Fast Groups Bypasser. `main`: its default branch became
    # `dev` on 2026-10-15, a tree without Power Lora Loader (queen-editor, madde 470).
    ("rgthree-comfy",             "https://github.com/rgthree/rgthree-comfy.git", "main"),
    # POSITIVE / NEGATIVE wildcard boxes, FaceDetailer, SAMLoader, switches
    ("ComfyUI-Impact-Pack",       "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git", None),
    # UltralyticsDetectorProvider (the face detector)
    ("ComfyUI-Impact-Subpack",    "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git", None),
    # easy int / easy float: Width, Height, Steps, Batch Size, Denoise
    ("ComfyUI-Easy-Use",          "https://github.com/yolain/ComfyUI-Easy-Use.git", None),
    # CLIPTextEncodeBREAK: Queen Editor's positive encoder, which splits the prompt on BREAK
    ("ComfyUI-ppm",               "https://github.com/pamparamm/ComfyUI-ppm.git", None),
    # The rest only sit in the graph's switched-off branches.
    ("ComfyUI-Custom-Scripts",    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git", None),
    ("ComfyUI_UltimateSDUpscale", "https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git", None),
    ("ComfyUI-KJNodes",           "https://github.com/kijai/ComfyUI-KJNodes.git", None),
]
# Impact-Pack lists sam2 straight from GitHub; pip builds it on the machine, torch and all, and the
# graph loads SAM's first version (queen-editor, madde 314). Impact-Pack imports sam2 only when present.
SKIPPED = {"git+https://github.com/facebookresearch/sam2"}

for name, url, branch in CUSTOM_NODES:
    target = f"{COMFY_ROOT}/custom_nodes/{name}"
    if os.path.exists(target) and os.listdir(target):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: klonlanıyor")
    # --recurse-submodules: UltimateSDUpscale vendors its upstream repo as a submodule.
    run(["git", "clone", "--depth", "1", "--recurse-submodules",
         *(["--branch", branch] if branch else []), url, target], f"clone {name}", timeout=180)
    if not os.listdir(target):
        raise RuntimeError(f"❌ {name}: klon sonrası klasör boş")
    req = f"{target}/requirements.txt"
    if os.path.exists(req):
        with open(req, encoding="utf-8") as f:
            lines = f.read().splitlines()
        # The kept copy sits beside the original: pip resolves a -r inside a list from where it is.
        kept = f"{target}/requirements.lora-trial.txt"
        with open(kept, "w", encoding="utf-8") as f:
            f.write("".join(f"{line}\n" for line in lines if line.strip() not in SKIPPED))
        run(["pip", "install", "-q", "-r", kept], f"pip install {name}", timeout=600)

log(f"ComfyUI ve {len(CUSTOM_NODES)} custom node hazır", "OK")

## 4) Modeller — ~15,9 GB

**Queen Editor'ün iki checkpoint'i**, Queen Editor'ün indirdiği Civitai sürümleri ve adlarıyla — LoRA kutusunun ikisinde de çalışması gerekiyor:

| Klasör | Dosya | Boyut | Civitai |
|---|---|---|---|
| `checkpoints/` | `nova3DCGXL_ilV90.safetensors` | 6,94 GB | [Nova 3DCG XL](https://civitai.com/models/715287), IL v9.0 |
| `checkpoints/` | `DasiwaIllustriousAnime_epitaphecstasy.safetensors` | 6,94 GB | [DaSiWa Illustrious](https://civitai.com/models/1034922), EpitaphEcstasy |

**Altı aday** — `loras/`'a, ayırt edilsin diye kendi adlarıyla (Civitai'deki adları `last-000008`, `Femboy`, … gibi). Neden bunlar: `instructions.md`.

| Grup | Dosya | Boyut | Civitai | Taban |
|---|---|---|---|---|
| femboy | `femboysXL_v1.safetensors` | 0,49 GB | [femboysXL](https://civitai.com/models/289006), v1.0 | SDXL 1.0 *(Animagine XL 3.0'da eğitilmiş)* |
| femboy | `femboy_illustrious_v1_3.safetensors` | 0,23 GB | [Femboy illustrious](https://civitai.com/models/1738161), v1.3 | Illustrious |
| femboy | `femboy_new_tos_il.safetensors` | 0,23 GB | [Femboy, Sissy, Feminine Male](https://civitai.com/models/1922853), IL | Illustrious |
| futa | `futa_flaccid_v3.safetensors` | 0,23 GB | [Flaccid Futa](https://civitai.com/models/2063309), v3.0 IL | Illustrious |
| futa | `futa_on_female_il_v2.safetensors` | 0,23 GB | [Futa on Female](https://civitai.com/models/1107758), Illustrious v2 | Illustrious |
| futa | `futa_on_male_il_v2.safetensors` | 0,23 GB | [Futa on Male / Futadom 2](https://civitai.com/models/891832), Illustrious v2 | Illustrious |

**Grafiğin yüz düzeltmesi** (FaceDetailer açık; Queen Editor'ün indirdiği iki dosya): `ultralytics/bbox/face_yolov9c.pt` ([Bingsu/adetailer](https://huggingface.co/Bingsu/adetailer), 52 MB) ve `sams/sam_vit_b_01ec64.pth` (Meta, 375 MB).

Önce **boş disk**: inecek dosyaların toplamı + 5 GiB pay yoksa hiçbir şey inmeden durur. Sonra **Civitai erişimi**: her dosyanın ilk 1 KB'ı çerezle istenir; çerez bayatsa hücre Civitai'nin kendi cevabıyla, ağır bir şey inmeden durur.

Civitai dosyaları `curl` ile (tek bağlantı) `<ad>.part`'a iner, **header'dan doğrulanır** (header dosyanın boyunu zaten kanıtlıyor), sonra gerçek yerine taşınır: ComfyUI yarım bir dosyayı gerçek adıyla görmez. `.part` yeniden çalıştırınca kendiliğinden ele alınır: yarımsa kaldığı yerden devam eder, tamsa yerine konur, header'ı bozuksa başı basılıp silinir ve baştan iner. `.pt` / `.pth` dosyaları küçük: her seferinde baştan iner, başı bir hata sayfası mı ve taban boyu geçiyor mu diye bakılır. Bozuk bir dosya yerine **konmaz**; yerindeki bozuk bir dosya silinmez, hücre başını basıp durur — yeniden denemeden önce onu elle sil.

**Sonunda adayların eğitim etiketleri basılır:** her LoRA'nın header'ı hangi modelde eğitildiğini ve eğitim açıklamalarında en sık geçen etiketleri taşıyabilir. Femboy illustrious v1.3'ün sayfası tetik yazmıyor (önceki sürümleri `Fboy` diyor); dosyanın kendi sayımı tetiği gösterirse workflow'daki tetiği ona göre düzelt.

In [ ]:
import os, json, glob, shutil, struct, subprocess, time, collections

# === What this notebook downloads ===
# Queen Editor's two checkpoints, its own Civitai versions under its own file names (queeneditor.ipynb,
# PHOTO_CHECKPOINTS): (model version, file name, models folder, bytes by Civitai's API, label).
CHECKPOINTS = [
    (2744564, "nova3DCGXL_ilV90.safetensors", "checkpoints", 6_938_530_484, "Nova 3DCG XL IL v9.0"),
    (3012006, "DasiwaIllustriousAnime_epitaphecstasy.safetensors", "checkpoints", 6_938_115_222,
     "DaSiWa Illustrious Anime EpitaphEcstasy"),
]
# The candidates, read from Civitai's API on 2026-10-27: (group, model version, file name, bytes by the
# API, label, strength, trigger). The group picks the frame its workflow opens with (section 5). A page
# that recommends no strength opens at 1.0, the loader's own. The trigger is what Queen Editor would put
# in front of the prompt (catalog.with_trigger): the page's trained words, less the ones a frame says
# itself -- counts (1boy, 1futa, 2girls) and body or act tags.
CANDIDATES = [
    # The user's pick to try. Trained on Animagine XL 3.0 (its header), an SDXL model, not Illustrious.
    ("femboy", 324974, "femboysXL_v1.safetensors", 489_722_956, "femboysXL v1.0", 1.0,
     "otoko no ko, crossdressing, trap, androgynous"),
    # The most downloaded femboy LoRA on the Illustrious base, and its newest version. v1.3's page lists
    # no trained word; v1.2 and V1 list Fboy. The summary below prints the file's own tag count.
    ("femboy", 2693790, "femboy_illustrious_v1_3.safetensors", 228_456_460, "Femboy illustrious v1.3",
     1.0, "Fboy"),
    # The second most downloaded femboy concept on Illustrious; its trigger is the page's one line.
    ("femboy", 2176350, "femboy_new_tos_il.safetensors", 228_476_020, "Femboy, Sissy, Feminine Male IL",
     1.0, "1femboy, lean male chest, feminine male"),
    # The futa body itself, the newest of the three (2026-02), retrained to a neutral style for stylised
    # checkpoints. Its page gives 0.4 - 0.8; this opens in the middle.
    ("futa", 2698395, "futa_flaccid_v3.safetensors", 228_460_108, "Flaccid Futa v3.0 IL", 0.6,
     "flaccid futa"),
    # The two most downloaded futa concepts on Illustrious, one with a woman and one with a man.
    ("futa", 1457484, "futa_on_female_il_v2.safetensors", 228_479_108, "Futa on Female IL v2", 1.0,
     "futa with female"),
    ("futa", 2066157, "futa_on_male_il_v2.safetensors", 228_482_300, "Futa on Male IL v2", 1.0,
     "futa on male, futa with male, futanari"),
]
CIVITAI = CHECKPOINTS + [(version_id, filename, "loras", size, label)
                         for _group, version_id, filename, size, label, _strength, _trigger in CANDIDATES]
# The graph's face fix, as Queen Editor fetches them: (address, models folder, file name, label, floor).
# A .pt/.pth file states no length of its own; the floor catches an error page or a cut-off file.
OPEN = [
    ("https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov9c.pt",
     "ultralytics/bbox", "face_yolov9c.pt", "Yüz dedektörü (yolov9c)", 40_000_000),
    ("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
     "sams", "sam_vit_b_01ec64.pth", "SAM ViT-B", 300_000_000),
]
# GiB the disk check keeps free past the models themselves; it also covers the two small files, whose
# floor rather than their size goes into the sum.
HEADROOM_GIB = 5

def target_of(folder, filename):
    return f"{COMFY_ROOT}/models/{folder}/{filename}"

def settled(path, label):
    """The header's verdict, or a RuntimeError showing how the file starts. A bad file is never
    passed on and never deleted: it stays where it is, for inspection -- and since a file in place is
    never fetched again, a re-run needs it deleted by hand first."""
    state, msg = check_safetensors(path)
    if state != "ok":
        raise RuntimeError(f"❌ {label}: {state} — {msg}\n{path}\n--- file head ---\n{head_text(path)}\n"
                           f"Yeniden denemeden önce bu dosyayı elle sil: {path}")
    return msg

# === Civitai (NOTEBOOK-STANDARD, section 4) ===
# civitai.red, where the cookie was issued: sent to .com it is cross-site and the answer is the login
# page. The login cookie only -- a ?token= API key authenticates as that key's account and gets 401 on
# a gated file. The cookie rides in curl's arguments, so an error prints curl's stderr, never the
# command line.
def civitai_url(version_id):
    return f"https://civitai.red/api/download/models/{version_id}"

def cookie_header():
    return f"Cookie: __Secure-civ-token={COOKIE_VALUE}"

PROBE = "/content/_probe.bin"

def civitai_probe(version_id, label):
    """The first KB of a Civitai file, asked with the cookie before anything heavy comes down: a stale
    or clipped cookie stops the run here, with Civitai's own answer. Exit 28 is curl's timeout, which
    --limit-rate and --max-time can bring on a good probe too (a store that ignores the range), so the
    code and the bytes decide. A safetensors file's header JSON starts at its ninth byte; a login page
    or an error JSON does not."""
    done = subprocess.run(["curl", "-sS", "-L", "--max-time", "20", "--limit-rate", "200k",
                           "-r", "0-1023", "-H", cookie_header(), "-w", "%{http_code}", "-o", PROBE,
                           civitai_url(version_id)], capture_output=True, text=True)
    if done.returncode not in (0, 28):
        raise RuntimeError(f"❌ {label}: Civitai probe — curl exit {done.returncode}\n{done.stderr.strip()}")
    code = (done.stdout or "").strip()[-3:]
    raw, said = b"", ""
    if os.path.exists(PROBE):
        with open(PROBE, "rb") as f:
            raw = f.read(10)
        said = head_text(PROBE)
        os.remove(PROBE)
    if code.startswith("2") and raw[8:10] == b'{"':
        log(f"{label}: Civitai erişimi OK", "OK")
        return
    raise RuntimeError(f"❌ {label}: Civitai probe — HTTP {code}\n"
                       f"--- Civitai yanıtı ---\n{said.strip() or '(boş gövde)'}"
                       + (f"\n--- curl ---\n{done.stderr.strip()}" if done.stderr.strip() else ""))

def civitai_fetch(version_id, filename, folder, label):
    target = target_of(folder, filename)
    if os.path.exists(target):
        log(f"{label}: zaten var ({settled(target, label)})")
        return
    url = civitai_url(version_id)
    part = target + ".part"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    # A .part an earlier run left is judged before curl touches it, as queen-editor's fetch does.
    state = None
    if os.path.exists(part):
        state, msg = check_safetensors(part)
        if state == "invalid":
            # NOTEBOOK-STANDARD section 3 keeps a bad file for inspection, but these bytes do not begin
            # with a safetensors header, so nothing appended to them can make a model of them, and
            # their head is printed right here. They go, and the file comes down fresh.
            log(f"{label}: .part bozuk ({msg}) — siliniyor, baştan iniyor\n"
                f"--- dosyanın başı ---\n{head_text(part)}", "WARN")
            os.remove(part)
        elif state == "ok":
            log(f"{label}: .part zaten tam ({msg}) — indirilmiyor")
        else:
            log(f"{label}: .part'tan devam ({msg})")
    start = time.perf_counter()
    before = os.path.getsize(part) if os.path.exists(part) else 0
    if state != "ok":
        log(f"{label}: Civitai'den iniyor")
        # One connection, by curl: Civitai redirects to its store, the B2 store answers 403 when the
        # cookie comes along, and curl drops it at the change of host where aria2c carries it. -C -
        # carries on from a partial .part. A stall stops it (under 1 MiB/s for 5 minutes); run()'s
        # 4 hours is the backstop.
        try:
            run(["curl", "-sS", "-L", "-C", "-", "--fail-with-body",
                 "--speed-limit", "1048576", "--speed-time", "300",
                 "-H", cookie_header(), "-o", part, url], label, timeout=4 * 3600)
        except RuntimeError as e:
            # The .part stays: what came down before the failure may be good, and the next run judges
            # it and resumes.
            raise RuntimeError(f"❌ {e}\n{url}\n--- .part'ın başı ---\n{head_text(part)}") from None
    msg = settled(part, label)
    os.replace(part, target)
    took, got = time.perf_counter() - start, os.path.getsize(target) - before
    log(f"{label}: indirildi ve doğrulandı — {took:.0f} sn, {got / took / 2**20:.1f} MB/s ({msg})", "OK")

def open_fetch(url, folder, filename, label, floor):
    """A small open file: fetched whole each time into a .part, checked, then put in place."""
    target = target_of(folder, filename)
    if os.path.exists(target):
        state, msg = check_binary(target, floor)
        if state != "ok":
            raise RuntimeError(f"❌ {label}: {state} — {msg}\n{target}\n--- file head ---\n"
                               f"{head_text(target)}\nYeniden denemeden önce bu dosyayı elle sil: {target}")
        log(f"{label}: zaten var ({msg})")
        return
    part = target + ".part"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    try:
        run(["curl", "-sS", "-L", "--fail-with-body", "--max-time", "1800", "-o", part, url],
            label, timeout=3600)
    except RuntimeError as e:
        raise RuntimeError(f"❌ {e}\n{url}\n--- .part'ın başı ---\n{head_text(part)}") from None
    state, msg = check_binary(part, floor)
    if state != "ok":
        raise RuntimeError(f"❌ {label}: {state} — {msg}\n{url}\n--- .part'ın başı ---\n{head_text(part)}")
    os.replace(part, target)
    log(f"{label}: indirildi ve doğrulandı ({msg})", "OK")

def trained_on(path, top=8):
    """What the lora's header says of its training: the model it was trained on, and the tags its
    captions count most -- the trainer's own words, so a trigger the page leaves out shows here."""
    with open(path, "rb") as f:
        meta = json.loads(f.read(struct.unpack("<Q", f.read(8))[0])).get("__metadata__") or {}
    counts = collections.Counter()
    for tags in json.loads(meta.get("ss_tag_frequency") or "{}").values():
        counts.update(tags)
    base = meta.get("ss_sd_model_name") or meta.get("ss_base_model_version") or "(yazmıyor)"
    return base, counts.most_common(top)

# === Free disk, before anything downloads ===
need = (sum(size for _, filename, folder, size, _ in CIVITAI
            if not os.path.exists(target_of(folder, filename)))
        + sum(floor for _, folder, filename, _, floor in OPEN
              if not os.path.exists(target_of(folder, filename))))
free = shutil.disk_usage("/content").free
log(f"İnecek: {need / 1024**3:.1f} GiB | Diskte boş: {free / 1024**3:.1f} GiB")
if free < need + HEADROOM_GIB * 1024**3:
    raise RuntimeError(f"❌ Disk yetmiyor: {need / 1024**3:.1f} GiB model + {HEADROOM_GIB} GiB pay "
                       f"gerekiyor, {free / 1024**3:.1f} GiB boş.")

# === Civitai access, before anything heavy ===
for version_id, filename, folder, _, label in CIVITAI:
    if not os.path.exists(target_of(folder, filename)):
        civitai_probe(version_id, label)

# === Downloads ===
for url, folder, filename, label, floor in OPEN:
    open_fetch(url, folder, filename, label, floor)
for version_id, filename, folder, _, label in CIVITAI:
    civitai_fetch(version_id, filename, folder, label)

# === Summary (reaching here means every model downloaded + validated) ===
print("")
for folder in ["checkpoints", "loras", "ultralytics/bbox", "sams"]:
    print(f"📂 {folder}/")
    for f in sorted(glob.glob(f"{COMFY_ROOT}/models/{folder}/*")):
        print(f"   {human(os.path.getsize(f)):>8s}  {os.path.basename(f)}")
print("\n🏷  Adayların eğitimi (header'larından):")
for group, _, filename, _, label, _, trigger in CANDIDATES:
    print(f"   [{group}] {label} — workflow'daki tetik: {trigger}")
    # Printed, not raised: every file is down and checked by now, and these lines are a hint for the
    # trigger -- a header they cannot read must not stop Run all before the workflows and ComfyUI.
    try:
        base, tags = trained_on(target_of("loras", filename))
    except Exception as e:
        print(f"      okunamadı — {type(e).__name__}: {e}")
        continue
    print(f"      eğitildiği model: {base}\n"
          f"      en sık etiketler: {', '.join(f'{tag} ({n})' for tag, n in tags) or '(sayım yok)'}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

## 5) Workflow'lar — Workflows paneline, aday başına bir tane

Queen Editor fotoğrafı bir **API grafiğiyle** çizer (`queen-editor/assets/workflow_api.json`): yalnız node'lar ve değerleri. Workflows paneli ise **arayüz biçiminde** bir workflow açar — node'ların yeri, kutuları, bağları. Queen Editor'ün grafiği nova-3dcg denemesinin arayüz grafiğinden (`collab-toolbox/photo_generator/nova-3dcg/workflow_manual.json`) export edildi: node numaraları aynı. Hücre ikisini de bu reponun `main`'inden okur — `GITHUB_TOKEN` varsa GitHub'ın API'sinden, yoksa açık adresinden — ve arayüz grafiğini Queen Editor'ünkine çevirir:

1. **Önce karşılaştırır.** API grafiğindeki her node arayüz grafiğinde aynı numarayla, açık (bypass değil) duruyor mu; her girdisi aynı node'dan mı bağlı (kapalı bir node'un içinden geçen bağ, export'un yaptığı gibi arkasındaki node'a kadar izlenir); elle yazılmış her değeri (adım, cfg, sampler, FaceDetailer'ın ayarları, negatif prompt, …) node'un kutularında var mı. Biri tutmazsa hücre durur ve hangisi olduğunu söyler: o zaman ekranda gördüğün Queen Editor'ün çizeceği olmazdı.
2. **Node türü farklıysa Queen Editor'ünkini alır** ve söyler. Bugün tek fark pozitif kodlayıcı: arayüz grafiğinde `CLIPTextEncode`, Queen Editor'de `CLIPTextEncodeBREAK` *(QueenAgent'ın `BREAK`'leri karakterleri ayırıyor)*.
3. **Her aday için Queen Editor'ün bir kareye yazdıklarını yazar:**

| Node | Ne yazılır |
|---|---|
| `45` Load Checkpoint | `nova3DCGXL_ilV90` — DaSiWa için kutudan çevir |
| `27` Power Lora Loader | **yalnız aday**, sayfanın gücüyle (önermeyende 1,0) — USNR yok, Queen Editor'de bir LoRA seçilince olduğu gibi |
| `3` POSITIVE | adayın tetiği + `, ` + grubunun karesi — femboy'da Kare F, futa'da Kare G (`instructions.md`) — Queen Editor'ün `with_trigger`'ı gibi |
| `40` Seed | sabit `20261027` — adaylar aynı gürültüden çizsin diye. Queen Editor her kareye kendi seed'ini verir |

**Negatif Queen Editor'deki gibi değil.** Queen Editor grafiğin kendi negatifini hiç kullanmaz: panelin negatif kutusuna yazılanı yazar — proje başına saklanır, boşsa boş gider. Örnek senaryonun bir negatifi yok, bu yüzden deneme grafiğin kendi negatifiyle çizer. Kendi projendeki negatifle görmek istersen onu NEGATIVE kutusuna yapıştır.

Workflow'un adı adayın dosya adı (`femboysXL_v1`, `futa_flaccid_v3`, …). Boyut (1024 × 1536, dikey), 28 adım, cfg 6, `euler_ancestral` ve FaceDetailer Queen Editor'ün grafiğindeki gibi kalır. Grafiğin kapalı dalları (upscale, HiresFix, ControlNet, …) kapalı kalır.

Hedefte aynı adla bir workflow zaten varsa dokunulmaz: UI'da aynı adla kaydettiğin bir grafik, hücre tekrar çalışınca ezilmez.

In [ ]:
# === Workflows -> ComfyUI's user workflows, one per candidate, from Queen Editor's photo graph ===
import os, json, copy, urllib.request, urllib.error

REPO = "AltanBaysal/Internal-tools"
# Queen Editor's photo graph, and the UI graph it was exported from (same node ids).
QE_GRAPH = "queen-editor/assets/workflow_api.json"
UI_GRAPH = "collab-toolbox/photo_generator/nova-3dcg/workflow_manual.json"
# The node ids comfy_photo_generator.py writes into.
PROMPT_NODE, LORA_NODE, SEED_NODE, MODEL_NODE = "3", "27", "40", "45"
WRITTEN = {PROMPT_NODE, LORA_NODE, SEED_NODE, MODEL_NODE}
# One seed for every candidate, so they differ by their lora alone.
SEED = 20261027
# The frame each group's workflows open with, the trigger in front: frames F and G of instructions.md,
# the sample scenario's doorway frame (frame 2) in the shape QueenAgent's build_prompts writes -- its
# woman recast as a femboy, and the same woman as a futanari.
FRAMES = {
    "femboy": (
        "score_9_up, score_9, score_8_up, masterpiece, best quality, raw, high quality, 4k, absurdres, "
        "1boy, solo, femboy, young man in his early 20s, androgynous, pale skin, long black hair, "
        "green eyes, slim body, flat chest, oversized long black t-shirt, black thong BREAK standing in "
        "doorway, door opening, light blush, soft flirty smile, looking at viewer, full body, apartment "
        "doorway, daytime, natural light, indoors"),
    "futa": (
        "score_9_up, score_9, score_8_up, masterpiece, best quality, raw, high quality, 4k, absurdres, "
        "1girl, solo, futanari, woman in her mid 20s, goth woman, pale skin, long black hair, "
        "green eyes, slim body, oversized long black t-shirt, black thong BREAK standing in "
        "doorway, door opening, light blush, soft flirty smile, looking at viewer, full body, apartment "
        "doorway, daytime, natural light, indoors"),
}
WORKFLOWS = f"{COMFY_ROOT}/user/default/workflows"

def fetched(path):
    """A JSON file of this repo's main, or a RuntimeError with what GitHub answered. With GITHUB_TOKEN
    through GitHub's API, which reads a private repo too; without it the raw address, which reads a
    public one. The token rides in a header only, and no message prints the request's headers."""
    if GITHUB_TOKEN:
        url = f"https://api.github.com/repos/{REPO}/contents/{path}?ref=main"
        request = urllib.request.Request(url, headers={
            "Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github.raw"})
    else:
        url = f"https://raw.githubusercontent.com/{REPO}/main/{path}"
        request = urllib.request.Request(url)
    try:
        with urllib.request.urlopen(request, timeout=60) as answer:
            body = answer.read()
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"❌ {url}\nHTTP {e.code}\n{e.read()[:2000].decode('utf-8', 'replace')}") from None
    except urllib.error.URLError as e:
        raise RuntimeError(f"❌ {url}\n{e.reason}") from None
    try:
        return json.loads(body)
    except ValueError as e:
        raise RuntimeError(f"❌ {url}: JSON değil — {e}\n{body[:2000].decode('utf-8', 'replace')}") from None

def differences(ui, api):
    """Where the UI graph would draw otherwise than Queen Editor's graph: an API node it lacks or has
    switched off, an input it does not link from the same node, a value it does not hold. The values
    of the nodes this cell writes are not compared; their links are.

    A value is looked for anywhere in the node's widgets_values, not at its widget's own position,
    which a UI file does not name; and Python counts True and False equal to 1 and 0. Both make the
    check looser, never stricter: it can miss a difference, but it cannot invent one."""
    nodes = {str(node["id"]): node for node in ui["nodes"]}
    links = {link[0]: link for link in ui["links"]}   # [id, from node, from slot, to node, to slot, type]

    def origin(link_id):
        """The node a link really comes from. A bypassed node hands on its input of the link's type
        -- the export follows it the same way, so Queen Editor's graph names the node behind it."""
        _, source, _, _, _, kind = links[link_id]
        node = nodes[str(source)]
        while node.get("mode") == 4:
            through = next((i["link"] for i in node.get("inputs", [])
                            if i.get("type") == kind and i.get("link") is not None), None)
            if through is None:
                return None
            node = nodes[str(links[through][1])]
        return str(node["id"])

    found = []
    for node_id, node in api.items():
        ours = nodes.get(node_id)
        if ours is None or ours.get("mode") != 0:
            found.append(f"{node_id} {node['class_type']}: arayüz grafiğinde yok ya da kapalı")
            continue
        linked = {i["name"]: origin(i["link"]) for i in ours.get("inputs", [])
                  if i.get("link") is not None}
        values = ours.get("widgets_values") or []
        for name, value in node["inputs"].items():
            if isinstance(value, list):
                if linked.get(name) != str(value[0]):
                    found.append(f"{node_id}.{name}: Queen Editor'de {value[0]}'dan bağlı, "
                                 f"arayüzde {linked.get(name) or 'bağsız'}")
            # Dicts are a loader's or a comparer's own state; "Select to add …" is a menu, not a value.
            elif (node_id not in WRITTEN and not isinstance(value, dict)
                  and not name.startswith("Select to add") and value not in values):
                found.append(f"{node_id}.{name}: Queen Editor'de {value!r}, arayüzde yok {values}")
    return found

qe_graph, ui_graph = fetched(QE_GRAPH), fetched(UI_GRAPH)
found = differences(ui_graph, qe_graph)
if found:
    raise RuntimeError("❌ Arayüz grafiği Queen Editor'ün grafiğinden ayrılmış — workflow'lar "
                       "Queen Editor'ün çizeceğini göstermezdi:\n  " + "\n  ".join(found))
log(f"Arayüz grafiği Queen Editor'ün grafiğini tutuyor ({len(qe_graph)} node)", "OK")

# A node of another class takes Queen Editor's -- the positive encoder today, CLIPTextEncodeBREAK --
# and loses the old class's pack id, which the frontend reads only to offer an install.
nodes = {str(node["id"]): node for node in ui_graph["nodes"]}
for node_id, node in qe_graph.items():
    ours = nodes[node_id]
    if ours["type"] != node["class_type"]:
        log(f"{node_id}: {ours['type']} → {node['class_type']}, Queen Editor'deki gibi")
        ours["type"] = node["class_type"]
        ours["title"] = node["_meta"]["title"]
        ours["properties"] = {"Node name for S&R": node["class_type"]}

def candidate_graph(filename, strength, prompt):
    """The UI graph as Queen Editor would draw this candidate on Nova: its lora alone in the loader,
    this prompt, one seed."""
    graph = copy.deepcopy(ui_graph)
    nodes = {str(node["id"]): node for node in graph["nodes"]}
    nodes[MODEL_NODE]["widgets_values"][0] = CHECKPOINTS[0][1]
    nodes[SEED_NODE]["widgets_values"][0] = SEED
    # ImpactWildcardProcessor: the text written, and the text it populates -- Queen Editor writes both.
    nodes[PROMPT_NODE]["widgets_values"][0:2] = [prompt, prompt]
    # rgthree builds one lora row per saved value carrying a "lora" key; the rest are its header and
    # button. Whatever lora rows the graph had go, and the candidate's goes right after the header.
    loader = [v for v in nodes[LORA_NODE]["widgets_values"] if not (isinstance(v, dict) and "lora" in v)]
    header = next(i for i, v in enumerate(loader)
                  if isinstance(v, dict) and v.get("type") == "PowerLoraLoaderHeaderWidget")
    loader.insert(header + 1, {"on": True, "lora": filename, "strength": strength, "strengthTwo": None})
    nodes[LORA_NODE]["widgets_values"] = loader
    return graph

os.makedirs(WORKFLOWS, exist_ok=True)
for group, _, filename, _, label, strength, trigger in CANDIDATES:
    name = filename[:-len(".safetensors")] + ".json"
    saved = f"{WORKFLOWS}/{name}"
    # A file already there may be the user's own save under the same name: a re-run leaves it alone.
    if os.path.exists(saved):
        log(f"Workflow zaten var, olduğu gibi bırakıldı: Workflows → {name[:-5]}")
        continue
    part = saved + ".part"
    with open(part, "w", encoding="utf-8") as f:
        json.dump(candidate_graph(filename, strength, f"{trigger}, {FRAMES[group]}"), f,
                  ensure_ascii=False)
    os.replace(part, saved)
    log(f"Workflow hazır: Workflows → {name[:-5]} — {label} @ {strength}, tetik: {trigger}", "OK")

## 6) Başlat + cloudflared Tünel

ComfyUI arka planda başlar (90 sn içinde `/system_stats` cevap vermezse log'un son 30 satırını basıp durur). Sonra ComfyUI'ye bir kez bildiği node'lar sorulur (`/object_info`): Queen Editor'ün grafiğindeki bir node yoksa — bir paket import'ta düştüyse — hücre hangisi olduğunu ve log'un sonunu basıp durur; UI'da kırmızı bir node olarak değil. Tünel linki basılır, sonra hücre **bilerek açık kalır** — biterse Colab runtime'ı boşta sayıp tüneli öldürür. ComfyUI'nin canlı log'u bu hücrede akar.

In [ ]:
import subprocess, time, urllib.request, re, os, json

if not os.path.isfile("/content/cloudflared"):
    run(["wget", "-q", "-O", "/content/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], "cloudflared")
    run(["chmod", "+x", "/content/cloudflared"], "chmod cloudflared")

# Re-run safety: kill the previous ComfyUI + tunnel before starting new ones
subprocess.run(["pkill", "-f", "main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(COMFY_LOG, "w")
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
                 cwd=COMFY_ROOT, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(COMFY_LOG).readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")
log(f"ComfyUI ayakta ({(i + 1) * 2}s)", "OK")

# Every class Queen Editor's graph runs, asked of ComfyUI once: a pack that failed its import shows
# here, by name, instead of as a red node in the UI. ComfyUI prints such a failure in its log.
with urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/object_info", timeout=60) as answer:
    known = json.loads(answer.read())
classes = {node["class_type"] for node in qe_graph.values()}
missing = sorted(classes - set(known))
if missing:
    print("".join(open(COMFY_LOG).readlines()[-40:]))
    raise RuntimeError(f"❌ ComfyUI bu node'ları tanımıyor: {', '.join(missing)} — yukarıdaki log'a bak")
log(f"Queen Editor'ün grafiğindeki {len(classes)} node türü ComfyUI'de var", "OK")

# cloudflared output goes to a file, not a pipe: an unread pipe fills up and blocks the process.
tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read()) if os.path.exists(tunlog) else None
    if m:
        link = m.group(0)
        break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 ComfyUI linki: {link}\n")
print("⬆️  Linke gir → sol kenarda Workflows:")
for group, _, filename, _, label, strength, trigger in CANDIDATES:
    print(f"    [{group:<6s}] {filename[:-len('.safetensors')]:<24s} — {label} @ {strength}, tetik: {trigger}")
print("\n🎲 Seed hepsinde 20261027: aynı kare, yalnız LoRA farklı")
print("🎨 DaSiWa için: Load Checkpoint → DasiwaIllustriousAnime_epitaphecstasy")
print("📝 Prompt: POSITIVE kutusu — öteki kareler instructions.md'de; tetik her zaman en önde")
print("🔁 LoRA'sız karşılaştırma: Power Lora Loader'da adayın satırını kapat (Queen Editor'de 'Boş')\n")

# === Keep the cell OPEN (critical) ===
# ComfyUI + the tunnel run in the background. If this cell ENDS, Colab can call the runtime idle
# and disconnect -> ComfyUI + link die. Streaming the log keeps the cell in the foreground and
# shows generation progress when Run is pressed in the UI.
# To stop: interrupt this cell (■) or Runtime -> Disconnect.
print("📡 ComfyUI çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", COMFY_LOG])
except KeyboardInterrupt:
    log("Hücre durduruldu — ComfyUI hâlâ arka planda çalışıyor (yeni link için bu hücreyi tekrar çalıştır).", "WARN")